# 3D Gaussian splatting mini project

This notebook runs on Google Colab, and also on your own Linux machine or VM with an NVIDIA GPU. The instructions are in the project PDF; this notebook runs its commands.

**On Google Colab: before you start**
1. Make sure the folder `gs-mini-project`, with this notebook in it, is in your Google Drive.
2. Upload the data files (`part1.tar.gz` to `part4.tar.gz`) to the folder `gs-mini-project` in your Google Drive, so that it contains the folder `code` (the code), `gsplat-1.5.3-cp312-cp312-linux_x86_64.whl` (gsplat, precompiled), `gs.ipynb`, and the four data files.
3. Runtime → Change runtime type → **T4 GPU** (an A100 could also work) and, under **Runtime version, choose 2026.07**. This matches the precompiled gsplat, so you do not wait for a long compile.
4. Run the cells from top to bottom.

**On your own machine or a VM instead:** install PyTorch (CUDA 12.x) and the build tools (`g++` and the CUDA toolkit with `nvcc`; see Sections 2.3 and 2.4 of the project PDF), start Jupyter in that environment, and put the folder `code` and the files `part*.tar.gz` in the folder of this notebook (or set `GS_DRIVE` to the folder that has them). (You don't need the notebook on your own machine: you can also run the commands of the project PDF in a terminal.) The precompiled gsplat is for Colab only: on your machine gsplat is compiled once, on its first use. Everything is stored in the folder `work/` next to the files.

**Storage on Colab.** Colab erases its own disk when a session ends, but not your Drive. So the code (including the files you edit) and your results stay in
your Drive folder, and only the data is copied to Colab's disk at the start of each session. If a session ends, start again from the top: the cells
skip what is already done, and your edits are never overwritten.

You edit the TODO files in the Colab file browser (left side, folder icon): `drive/MyDrive/gs-mini-project/code/part2/depth_losses.py` and so on.
Training always runs as a new process, so it picks up your latest saved edit.

## 1. GPU and Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
try:
    from google.colab import drive
    drive.mount('/content/drive'); IN_COLAB = True
    DRIVE = '/content/drive/MyDrive/gs-mini-project'    # change this if you used another folder name
    DATA = '/content/data'                              # local disk: fast, but erased when the session ends
    RUNS = '/content/runs'                              # local disk too: checkpoints are large (see the sync cell)
except ImportError:                                     # not on Colab: everything next to this notebook
    IN_COLAB = False
    DRIVE = os.environ.get('GS_DRIVE', os.getcwd())
    DATA = f'{DRIVE}/work/data'; RUNS = f'{DRIVE}/work/runs'
print(os.listdir(DRIVE))

## 2. Go to the code
The code is the folder `code` in your Drive folder. You edit its files in place, and they stay there between sessions.

In [ ]:
PKG = f'{DRIVE}/code'
assert os.path.exists(f'{PKG}/gsmini'), 'the code folder was not found: is the folder code (with gsmini/ inside it) in your Drive folder, and is it the one from the course?'
%cd {PKG}

## 3. Install and check
On Colab, PyTorch is already installed and is left alone, and with the precompiled gsplat from the course folder (runtime 2026.07) there is nothing to compile. Otherwise gsplat compiles its CUDA code the first time it runs. On Colab the compiled code is kept in your Drive folder, so later sessions skip the wait.

In [ ]:
os.environ['GS_DATA_DIR'] = DATA
import sys, torch
if IN_COLAB:
    os.environ['MAX_JOBS'] = '2'      # Colab has 2 CPU cores and about 12 GB RAM: more parallel compile jobs only make it swap
    os.environ['VERBOSE'] = '1'       # print gsplat's compile progress ([n/29] lines)
    cc = torch.cuda.get_device_capability()
    os.environ['TORCH_EXTENSIONS_DIR'] = f"{DRIVE}/torch_ext/sm{cc[0]}{cc[1]}_torch{torch.__version__.split('+')[0]}"
    if os.path.exists(f"{DRIVE}/torch_ext.tar") and not os.path.exists(os.environ['TORCH_EXTENSIONS_DIR']):
        !tar xf "{DRIVE}/torch_ext.tar" -C "{DRIVE}"      # the precompiled gsplat from the course folder (if it fits this GPU and PyTorch)
    WHEEL = f"{DRIVE}/gsplat-1.5.3-cp312-cp312-linux_x86_64.whl"          # gsplat precompiled for Colab runtime 2026.07 (T4 and A100)
    if sys.version_info[:2] == (3, 12) and torch.__version__.startswith('2.11.'):
        if os.path.exists(WHEEL):
            %pip install -q "{WHEEL}"
            print('installed the precompiled gsplat: no compile needed')
    else:
        print(f"This runtime has Python {sys.version_info[0]}.{sys.version_info[1]} and PyTorch {torch.__version__}.")
        print("The precompiled gsplat is for runtime version 2026.07. Switch with: Runtime > Change runtime type > Runtime version > 2026.07,")
        print("then run this notebook again from the top. (You can also continue: gsplat is then compiled, which takes a long time.)")
else:
    print("Not on Colab: gsplat is compiled once, on its first use (needs nvcc and g++; check_setup.py below tells you what is missing).")
REQ = 'requirements-colab.txt' if IN_COLAB else 'requirements.txt'
%pip install -q -r {REQ}
!python check_setup.py
from gsmini.report import results_table, show_renders, compare_renders
def sync():                                              # copy metrics and renders (not the big checkpoints) to your Drive
    !rsync -a --exclude gaussians.pt --exclude splat.ply "{RUNS}/" "{DRIVE}/runs/"

**Results.** Run `sync()` at the end of a part to copy your metrics and renders to Drive: the local disk is erased when the session ends, and the model checkpoints are too big to keep.

---
## Part 1: set up and train a baseline

Copy the data to Colab's local disk (skipped if it is already there). The instructions of this part are in Section 5 of the project PDF.

In [ ]:
if not os.path.exists(f'{DATA}/part1_garden'):
    !mkdir -p "{DATA}" && tar xzf "{DRIVE}/part1.tar.gz" -C "{DATA}"
!ls "{DATA}/part1_garden"

Train the baseline:

In [ ]:
!python part1/train.py --out "{RUNS}/p1_base"

The table lists the metrics of the run. Each saved image shows the **ground truth** photo next to **your render** from the same camera.

In [ ]:
results_table([f"{RUNS}/p1_base"])
show_renders(f"{RUNS}/p1_base", "val", n=2)

A fly-through video of your model:

In [ ]:
!python part1/render_video.py --run "{RUNS}/p1_base" --frames 120
from IPython.display import Video
Video(f"{RUNS}/p1_base/flythrough.mp4", embed=True, width=640)

---
## Part 2: sparse views and a monocular depth prior

Copy the data to Colab's local disk (skipped if it is already there). The instructions of this part are in Section 6 of the project PDF.

In [ ]:
if not os.path.exists(f'{DATA}/part2_room'):
    !mkdir -p "{DATA}" && tar xzf "{DRIVE}/part2.tar.gz" -C "{DATA}"
!ls "{DATA}/part2_room"

Open `part2/depth_losses.py`, implement the L1 loss, save, and run the check. Until you do, it prints which function is still missing.

In [ ]:
!python part2/check_losses.py

Baseline and depth prior:

In [ ]:
!python part2/train.py --loss none --out "{RUNS}/p2_base"
!python part2/train.py --loss l1 --out "{RUNS}/p2_l1"

The table lists the metrics of each run. Below it, the same held-out views side by side: the **ground truth** photo, the render of the baseline, and the render with the depth prior.

In [ ]:
results_table([f"{RUNS}/p2_base", f"{RUNS}/p2_l1"])
compare_renders([f"{RUNS}/p2_base", f"{RUNS}/p2_l1"], "val", n=3, labels=["baseline", "+ depth prior"])

---
## Part 3: a better training recipe for a street scene

Copy the data to Colab's local disk (skipped if it is already there). The instructions of this part are in Section 7 of the project PDF.

In [ ]:
if not os.path.exists(f'{DATA}/part3_wanderland'):
    !mkdir -p "{DATA}" && tar xzf "{DRIVE}/part3.tar.gz" -C "{DATA}"
!ls "{DATA}/part3_wanderland"

Baseline:

In [ ]:
!python part3/train.py --out "{RUNS}/p3_base"

In [ ]:
results_table([f"{RUNS}/p3_base"])

**Your experiments.** Change the settings of `part3/train.py` (all the flags are described in Section 7 of the project PDF and in `python part3/train.py --help`) and compare each run with the baseline. You need to run at least three different experiments, each with different settings, and report the image metrics for each one. Copy these cells and change the flags; give every run its own `--out` name:

In [ ]:
!python part3/train.py --refine-every 500 --out "{RUNS}/p3_try1"

**Your best run**, with the flags you found (the cell below is the baseline until you add your flags):

In [ ]:
!python part3/train.py --out "{RUNS}/p3_best"  # add your flags here

Check the target of this part (at least +0.75 dB above the baseline). The baseline has to be trained with the default number of steps; your best run can use any number:

In [ ]:
!python part3/check_goal.py --baseline "{RUNS}/p3_base" --mine "{RUNS}/p3_best"

The table lists the metrics of the runs. Below it, the same held-out views side by side: the **ground truth** photo, then the render of each run.

In [ ]:
results_table([f"{RUNS}/p3_base", f"{RUNS}/p3_best"])
compare_renders([f"{RUNS}/p3_base", f"{RUNS}/p3_best"], "val_offtraj", n=3, labels=["baseline", "your best"])

---
## Part 4: real robot data and LiDAR

Copy the data to Colab's local disk (skipped if it is already there). The instructions of this part are in Section 8 of the project PDF.

In [ ]:
if not os.path.exists(f'{DATA}/part4_robot'):
    !mkdir -p "{DATA}" && tar xzf "{DRIVE}/part4.tar.gz" -C "{DATA}"
!ls "{DATA}/part4_robot"

**The baseline** is plain 3DGS, started from randomly placed Gaussians. Train it. Your goal is to reach a `val_pass2` LPIPS of at most **0.58** (the baseline gets about 0.62). **Lower LPIPS is better.**

In [ ]:
!python part4/train.py --out "{RUNS}/p4_base"

In [ ]:
results_table([f"{RUNS}/p4_base"])

**Your improvement.** Change the code in `part4/` however you like (for example `train.py`), using the training photos and the LiDAR scans of drive 1, then run it. `part4/train.py` can start from your own initial Gaussians: add `--init file --init-file {DRIVE}/my_init.npz` (an `.npz` with `xyz` [N, 3] and `rgb` [N, 3] uint8) to the command.

In [ ]:
!python part4/train.py --out "{RUNS}/p4_mine"

In [ ]:
!python part4/check_goal.py --mine "{RUNS}/p4_mine"

The table lists the metrics of each run. Below it, the same held-out views from the second drive side by side: the **ground truth** photo, the render of the baseline, and the render of your model.

In [ ]:
results_table([f"{RUNS}/p4_base", f"{RUNS}/p4_mine"])
compare_renders([f"{RUNS}/p4_base", f"{RUNS}/p4_mine"], "val_pass2", n=3, labels=["baseline", "yours"])

---
## Saving your results
Copies the metrics and renders of your runs to the `runs/` folder in your Drive folder, so they survive the end of the Colab session.

In [ ]:
sync()   # keep your results on Drive